# 03 — Training theta on a DAG dataset

This notebook turns the prototype into a small learning loop.
The goal is to optimize a positive coefficient vector `theta` that parameterizes
\[
H_	heta(P) = \sum_{k=1}^K 	heta_k P^k.
\]

This is the notebook to keep as the main training playground while the reusable code lives in `src/`.

In [ ]:
import pathlib
import sys

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt

from l2o_cyber_defense import (
    make_dag_dataset,
    control_mask_from_dag,
    dag_target_w,
    sample_objective,
    evaluate_objective,
)
from l2o_cyber_defense.utils import theta_pos

## Hyperparameters

In [ ]:
torch.manual_seed(2)

K = 5
Ksum = 20
alpha = 0.5

epochs = 60
lr = 5e-2
grad_clip = 1.0

train_mats = 24
test_mats = 12
nmin, nmax = 12, 24
edge_prob = 0.30
rho = 0.90

## Dataset

In [ ]:
Ps_train, train_sizes = make_dag_dataset(
    num_mats=train_mats,
    n_min=nmin,
    n_max=nmax,
    rho=rho,
    edge_prob=edge_prob,
)
Ps_test, test_sizes = make_dag_dataset(
    num_mats=test_mats,
    n_min=nmin,
    n_max=nmax,
    rho=rho,
    edge_prob=edge_prob,
)

print("Train sizes:", train_sizes)
print("Test sizes :", test_sizes)

## Optimize `theta`

In [ ]:
raw_theta = torch.nn.Parameter(torch.zeros(K))
optimizer = optim.Adam([raw_theta], lr=lr)

train_hist = []
test_hist = []
theta_hist = []

for epoch in range(epochs):
    optimizer.zero_grad()
    theta = theta_pos(raw_theta)

    losses = []
    for P in Ps_train:
        control_mask = control_mask_from_dag(P)
        w = dag_target_w(P)
        obj = sample_objective(
            P=P,
            theta=theta,
            w=w,
            alpha=alpha,
            K=K,
            Ksum=Ksum,
            control_mask=control_mask,
        )
        losses.append(-obj)

    loss = torch.stack(losses).mean()
    loss.backward()
    torch.nn.utils.clip_grad_norm_([raw_theta], grad_clip)
    optimizer.step()

    theta_eval = theta_pos(raw_theta).detach()
    train_mean, _ = evaluate_objective(Ps_train, theta_eval, alpha, K, Ksum, dag_mode=True)
    test_mean, _ = evaluate_objective(Ps_test, theta_eval, alpha, K, Ksum, dag_mode=True)

    train_hist.append(train_mean)
    test_hist.append(test_mean)
    theta_hist.append(theta_eval.cpu().numpy())

    if epoch % 10 == 0 or epoch == epochs - 1:
        print(f"epoch={epoch:03d} train={train_mean:.4f} test={test_mean:.4f} theta={theta_eval.tolist()}")

## Learning curves

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(train_hist, label="train objective", linewidth=2)
plt.plot(test_hist, label="test objective", linewidth=2)
plt.xlabel("epoch")
plt.ylabel("objective")
plt.legend()
plt.tight_layout()
plt.show()

## Learned coefficients

In [ ]:
theta_hist_np = np.array(theta_hist)

plt.figure(figsize=(7, 4))
for k in range(theta_hist_np.shape[1]):
    plt.plot(theta_hist_np[:, k], label=fr"$\theta_{k+1}$")
plt.xlabel("epoch")
plt.ylabel("value")
plt.legend(ncol=2)
plt.tight_layout()
plt.show()

print("Final theta:", theta_hist_np[-1].tolist())

## Comments

This notebook is intentionally simple.
The next useful refactor would be to move this loop into `scripts/train_theta.py`
and to add experiment configuration files once the modeling choices stabilize.